# MCTS for Reasoning（蒙特卡洛树搜索）

**难度：** Hard

**函数签名：** `mcts_step(values, visit_counts, parent_visits, c_puct=1.414) -> (int, Tensor, Tensor)`

**参数：**
- `values` — N 个子节点的价值估计 (N,)
- `visit_counts` — 每个子节点的访问次数 (N,)
- `parent_visits` — 父节点总访问次数
- `c_puct` — 探索常数（默认 √2 ≈ 1.414）

**返回：** `(selected_idx, updated_values, updated_visits)`

**算法：** 选择（UCB1）→ 模拟（随机 rollout）→ 反向传播（滑动均值更新）

In [ ]:
import torch
import math

In [ ]:
# ✅ SOLUTION

def mcts_step(values, visit_counts, parent_visits, c_puct=1.414):
    import math as _math

    # ---- Step 1: 计算 UCB1 分数 ----
    # UCB(i) = Q(i) + c_puct * √(log(N_parent + 1) / (n_i + 1))
    # Q(i): 节点 i 的当前价值估计（利用项）
    # 第二项: 探索项，访问次数少的节点被鼓励探索
    #   - log(N_parent + 1): 父节点访问越多，探索项整体越小
    #   - n_i + 1: 访问次数少的节点，分母小，探索项大
    #   - c_puct: 探索常数，控制探索/利用权衡
    #   - √2 ≈ 1.414 是理论最优值（来自 UCB1 算法）
    exploration = c_puct * torch.sqrt(
        torch.log(torch.tensor(parent_visits + 1, dtype=torch.float32)) /
        (visit_counts.float() + 1)
    )
    ucb = values + exploration

    # ---- Step 2: 选择 UCB 最高的节点 ----
    # argmax 返回最高分数的索引
    selected_idx = int(ucb.argmax().item())

    # ---- Step 3: 模拟（Rollout）----
    # 为选中节点生成一个随机价值估计 [0, 1]
    # 实际 MCTS 中这里会调用神经网络评估或随机模拟到终局
    rollout_value = torch.rand(1).item()

    # ---- Step 4: 反向传播（滑动均值更新）----
    # Q_new = (Q_old * n + rollout) / (n + 1)
    # 这是增量均值公式: 新均值 = (旧均值 * 旧计数 + 新值) / 新计数
    # n 是旧的访问次数（更新前的值）
    n = visit_counts[selected_idx].item()
    new_value = (values[selected_idx].item() * n + rollout_value) / (n + 1)

    # 创建副本避免修改原始张量
    updated_values = values.clone()
    updated_visits = visit_counts.clone()
    updated_values[selected_idx] = new_value
    updated_visits[selected_idx] += 1

    return selected_idx, updated_values, updated_visits

In [ ]:
torch.manual_seed(42)
num_children = 5
values = torch.zeros(num_children)
visits = torch.zeros(num_children, dtype=torch.long)
parent_visits = 0

print("Running 5 MCTS steps:")
print(f"{'Step':>4}  {'Selected':>8}  {'Visit counts'}")
for step in range(5):
    idx, values, visits = mcts_step(values, visits, parent_visits)
    parent_visits += 1
    print(f"{step+1:>4}  {idx:>8}  {visits.tolist()}")

print(f"\nTotal visits: {visits.sum().item()} (expected 5)")
print(f"All nodes visited at least once: {(visits > 0).all().item()}")

In [ ]:
from torch_judge import check
check("mcts_search")

## 📝 核心思路总结

1. **UCB1 公式**：利用项 Q(i) + 探索项 c_puct*√(log(N)/n)，平衡探索与利用
2. **滑动均值更新**：`Q_new = (Q_old*n + val)/(n+1)`，增量式更新避免存储所有历史
3. **c_puct = √2**：理论最优探索常数
4. **三步流程**：选择 → 模拟 → 反向传播